# HOWTO: Simple Trees used for Search-based AI in Python


To understand AI search trees, you need to understand how tree data structures in general work. If you are not sure, then you can watch 
the video [Binary Search Trees Python Tutorial](https://www.youtube.com/watch?v=pkYVOmU3MgA)
from the course Data Structures and Algorithms in Python - Full Course for Beginners.

Search trees used in AI slightly differ from the usual binary search trees taught in data structures courses. Here are some key differences:

* AI search trees are dynamically built during search. Branches no longer needed during 
    search may be removed to save memory (e.g., in DFS).
* AI search trees are typically not binary trees, but nodes can have many children
    (often representing actions).
* For AI search trees we typically do not use links from parents to children, but 
    child notes have a link (references in Python) to their parent. The reason is that going down the tree, new nodes 
    are creates so we automatically have a reference. Links are only used to go back up the tree.

Here is an implementation of the basic node structure for an AI search tree (see Fig 3.7 on page 73). The state in the example is a position represented as `(row, col)`. I have added a method that extracts the path from the root node to the current node. It can be used to get the path when the search is completed.

In [14]:
class Node:
    def __init__(self, pos, parent, action, cost):
        self.pos = tuple(pos)    # trạng thái; vị trí có dạng (row, col)
        self.parent = parent     # tham chiếu tới nút cha. None nghĩa là nút gốc.
        self.action = action     # hành động dùng trong hàm chuyển trạng thái (nút gốc có giá trị None)
        self.cost = cost         # với tìm kiếm chi phí đều (uniform cost) đây là độ sâu. Đây cũng là g(n) trong tìm kiếm A*

    def __str__(self):
        """in nút ra dưới dạng dễ đọc"""
        return f"Node - pos = {self.pos}; action = {self.action}; cost = {self.cost}"
    
    def get_path_from_root(self):
        """trả về các nút trên đường đi từ nút gốc đến nút hiện tại."""
        node = self
        path = [node]
    
        while not node.parent is None:
            node = node.parent
            path.append(node)
        
        path.reverse()
        
        return(path)

## Ví dụ

Tạo nút gốc và thêm nút ứng với hành động đi về phía đông (E) với chi phí 1. Ví dụ này cũng minh họa cách quản lý bộ nhớ bằng bộ đếm tham chiếu (reference count) và cơ chế thu gom rác (garbage collection) (xem [Python references](https://www.pythontutorial.net/advanced-python/python-references/)).

In [15]:
# import sys to access reference counts
import sys

Tạo nút gốc tại vị trí (0,0). Không cần hành động nào để đến trạng thái của nút gốc, và chi phí đường đi để đến đó là 0.

In [16]:
root = Node(pos = (0,0), parent = None, action = None, cost = 0)
print("root:", root)
print("root (reference):", repr(root))
# Note: -1 is used because passing root to the getrefcount function adds a reference
print("Root's reference count for garbage collection:", sys.getrefcount(root) - 1) 

root: Node - pos = (0, 0); action = None; cost = 0
root (reference): <__main__.Node object at 0x0000013EAA888C20>
Root's reference count for garbage collection: 1


Go east with cost 1 (from the parent root).

In [17]:
node2 = Node(pos = (0,1), parent = root, action = "E", cost = 1)
print("node2:", node2)
print("parent of node2: ", node2.parent)
print("parent of node2 (reference): ", repr(node2.parent))
print("Root is now referenced two times (root variable, parent of node2). Reference count for garbage collection (root node):", sys.getrefcount(root) -1 )

node2: Node - pos = (0, 1); action = E; cost = 1
parent of node2:  Node - pos = (0, 0); action = None; cost = 0
parent of node2 (reference):  <__main__.Node object at 0x0000013EAA888C20>
Root is now referenced two times (root variable, parent of node2). Reference count for garbage collection (root node): 2


Lưu ý rằng nút gốc sẽ không bị thu gom rác chừng nào node2 vẫn còn giữ tham chiếu tới nó (qua thuộc tính parent).

In [18]:
root = None
print(root)
print("parent of node2 (reference to root node): ", node2.parent)
print("parent of node2 (reference to root node): ", repr(node2.parent))
print("Reference count for garbage collection (root node):", sys.getrefcount(node2.parent)-1)

None
parent of node2 (reference to root node):  Node - pos = (0, 0); action = None; cost = 0
parent of node2 (reference to root node):  <__main__.Node object at 0x0000013EAA888C20>
Reference count for garbage collection (root node): 1


Path from root to node2

In [19]:
path = node2.get_path_from_root()
print("References:", path)
print("Positions:", [n.pos for n in path]) #dãy trạng thái
print("Actions:", [n.action for n in path]) #dãy hành động
print("Path Cost:", [n.cost for n in path])

References: [<__main__.Node object at 0x0000013EAA888C20>, <__main__.Node object at 0x0000013EAA5D6FD0>]
Positions: [(0, 0), (0, 1)]
Actions: [None, 'E']
Path Cost: [0, 1]


- Bắt đầu từ nút hiện tại `node2`, lần theo thuộc tính parent để đi ngược lên, cho tới khi gặp nút gốc `parent is None`.

- Mỗi bước thêm nút vào danh sách path, nên danh sách đang có thứ tự từ nút hiện tại về nút gốc. Gọi path. `reverse()` để đảo lại thành thứ tự từ gốc đến nút hiện tại.

- Từ danh sách nút, lấy ra dãy trạng thái `[n.pos for n in path] = [(0,0), (0,1)]` và dãy hành động `[n.action for n in path] = [None, 'E']`.

=> xuất phát từ `(0,0)`, thực hiện hành động E (đi sang phía Đông) thì đến (0,1). Nút gốc có hành động None vì không cần hành động nào để đến đó.

Once we delete the reference to node2, the reference count for all nodes goes to zero and the whole tree is exposed to garbage collection.

In [20]:
node2 = None 

**Vì sao cây tìm kiếm AI thường được tạo động**
- Vì Không gian trạng thái của cây tìm kiếm AI rất lớn, có thể là vô hạn, nên không thể dựng sẵn toàn bộ cây trong bộ nhớ. Thuật toán chỉ sinh nút con khi mở rộng một nút trong lúc tìm kiếm. Những nhánh không còn cần có thể bỏ đi để tiết kiệm bộ nhớ, ví dụ như DFS sau khi quay lui. Khi không còn biến nào tham chiếu tới một nhánh, Python tự thu gom rác nhánh đó như cell `node2 = None`

**Vì sao không nhất thiết là cây nhị phân**
- vì ở cây nhị phân, một node có 2 node con cố định. Còn ở cây tìm AI, số node con có thể nhiều hơn chẳng hạn như N, E, S, W. Ở cây tìm kiếm AI không lưu danh sách con `left/right` như cây nhị phân, mà chỉ liên kết từ con lên cha (parent). Vì vậy số lượng node con không bị giới hạn.
